In [0]:
%pip install kafka-python confluent-kafka websocket-client

In [0]:
%restart_python
dbutils.library.restartPython()

In [0]:
BOOTSTRAP_SERVERS = "pkc-oxqxx9.us-east-1.aws.confluent.cloud:9092"

TOPIC = "crypto-transaction"

API_KEY = "E7FIZIBKNYJXFO4K"
API_SECRET = "cflt1GuGMjLlEF431MS+O0gIcFHAxOgMA9p+KRV5j3IARQQ8cFFNPAtm9+yXkjbQ"

In [0]:
raw_stream_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
        .option("subscribe", TOPIC)

        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")

        .option(
            "kafka.sasl.jaas.config",
            f"""
            kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required
            username="{API_KEY}"
            password="{API_SECRET}";
            """
        )

        .option("startingOffsets", "earliest")
        .load()
)

In [0]:
test_query = (
    raw_stream_df.writeStream
        .format("memory")
        .queryName("crypto_kafka_test")
        .outputMode("append")
        .trigger(availableNow=True)
        .option(
            "checkpointLocation",
            "s3a://sebastian-crypto-lakehouse/checkpoints/test_kafka_v6/"
        )
        .start()
)

In [0]:
test_query.awaitTermination()

In [0]:
spark.sql("SELECT * FROM crypto_kafka_test").show(truncate=False)